# 03 — GitHub Mordred feature store, LazyPredict ve final regression modeli

Bu sürüm gerçek GitHub repo yapısıyla uyumludur ve Mordred feature matrisindeki
`NaN`, `+inf` ve `-inf` değerlerini güvenli biçimde işler.

- Tamamen boş descriptorlar çıkarılır.
- Eksik oranı yüksek descriptorlar çıkarılır.
- Median imputer yalnızca eğitim setinde fit edilir.
- Aynı imputer test setine uygulanır.
- Feature sırası korunur.
- LazyPredict ve final model aynı temizlenmiş matrisi kullanır.

## Notebook akışı

```text
Paket ve ayarlar
      ↓
GitHub repo ağacında Mordred CSV arama
      ↓
Feature / target ayrımı
      ↓
Train / test split
      ↓
LazyPredict veya geçerli cache
      ↓
Final SHAP uyumlu ağaç modeli
      ↓
Holdout + 5-fold CV
      ↓
Google Drive kayıtları
```

> Hücreleri yukarıdan aşağıya bir kez çalıştırınız.  
> LazyPredict hücresi, geçerli cache bulunduğunda pahalı model taramasını tekrar etmez.

## Hücre 1 — Paket kontrolü

In [ ]:
import importlib.util
# Paketlerin aktif Python ortamında kurulu olup olmadığını kontrol etmek için kullanılır.

import subprocess
# Eksik paketleri aktif Python ortamına kurmak için kullanılır.

import sys
# Colab oturumunda kullanılan aktif Python yorumlayıcısını belirlemek için kullanılır.


REQUIRED_PACKAGES = [
    ("numpy", "numpy"),
    # Sayısal matris ve indeks işlemleri için kullanılır.

    ("pandas", "pandas"),
    # CSV okuma, sonuç tabloları ve veri yönetimi için kullanılır.

    ("requests", "requests"),
    # GitHub API ve RAW bağlantılarına erişmek için kullanılır.

    ("matplotlib", "matplotlib"),
    # Gerçek-tahmin grafiğini çizmek için kullanılır.

    ("sklearn", "scikit-learn"),
    # Train/test, modeller, CV ve metrikler için kullanılır.

    ("joblib", "joblib"),
    # Eğitilmiş modeli Google Drive'a kaydetmek için kullanılır.

    ("lazypredict", "lazypredict==0.3.0"),
    # Temel regression algoritmalarını karşılaştırmak için kullanılır.
]
# Gerekli paketlerin import ve pip adları tanımlanır.


for import_name, pip_name in REQUIRED_PACKAGES:
    # Paketler sırayla kontrol edilir.

    if importlib.util.find_spec(import_name) is None:
        # Paket aktif ortamda yoksa kurulum yapılır.

        print(f"Kuruluyor: {pip_name}")
        # Kurulacak paket kullanıcıya gösterilir.

        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                pip_name,
            ]
        )
        # Paket notebookun kullandığı aktif Python ortamına kurulur.

    else:
        # Paket zaten kuruluysa tekrar kurulum yapılmaz.

        print(f"Zaten kurulu: {import_name}")
        # Mevcut paket bilgisi yazdırılır.


print("✅ CHECKPOINT 03.1: Paket kontrolü tamamlandı.")

## Hücre 2 — Kütüphanelerin içe aktarılması

In [ ]:
from pathlib import Path
# Google Drive ve çıktı dosya yollarını yönetmek için kullanılır.

from io import BytesIO
# GitHub'dan indirilen CSV içeriğini bellekte pandas'a aktarmak için kullanılır.

import hashlib
# Veri ve ayar imzası oluşturarak cache doğrulaması yapmak için kullanılır.

import json
# Model bundle ve cache metadata dosyalarını kaydetmek için kullanılır.

import warnings
# Uzun model uyarılarını azaltmak için kullanılır.

warnings.filterwarnings("ignore")
# Workshop ekranını kalabalıklaştıran genel uyarılar gizlenir.

import joblib
# Eğitilmiş final modeli dosyaya kaydetmek için kullanılır.

import matplotlib.pyplot as plt
# Gerçek ve tahmin edilen değerleri çizmek için kullanılır.

import numpy as np
# Sayısal matris ve indeks işlemleri için kullanılır.

import pandas as pd
# CSV okuma, feature matrisi ve sonuç tabloları için kullanılır.

import requests
# GitHub API ve RAW dosya bağlantılarına istek göndermek için kullanılır.

from IPython.display import display
# DataFrame tablolarını Colab içinde göstermek için kullanılır.

from google.colab import drive
# Bütün sonuçları Google Drive'a kaydetmek için kullanılır.

from lazypredict.Supervised import LazyRegressor
# Birden fazla regression algoritmasını aynı split üzerinde karşılaştırmak için kullanılır.

from sklearn.ensemble import (
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    RandomForestRegressor,
)
# Final SHAP uyumlu ağaç modeli adayları içe aktarılır.

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
# Regression performans metrikleri içe aktarılır.

from sklearn.model_selection import (
    KFold,
    cross_validate,
    train_test_split,
)
# Train/test ayrımı ve 5-fold çapraz doğrulama araçları içe aktarılır.


print("Pandas sürümü:", pd.__version__)
# Kullanılan pandas sürümü kontrol amacıyla gösterilir.

print("✅ CHECKPOINT 03.2: Kütüphaneler başarıyla içe aktarıldı.")

from sklearn.impute import SimpleImputer
# Mordred descriptorlarındaki eksik değerleri yalnızca eğitim medyanıyla doldurmak için kullanılır.

## Hücre 3 — Google Drive bağlantısı

In [ ]:
drive.mount(
    "/content/drive",
    force_remount=False,
)
# Google Drive standart Colab dizinine bağlanır.


print("✅ CHECKPOINT 03.3: Google Drive bağlantısı hazır.")

## Hücre 4 — Target, GitHub ve model ayarları

In [ ]:
TARGET_ID = "CHEMBL206"
# İşlenecek ChEMBL Target ID burada değiştirilir.


GITHUB_OWNER = "MOL-FEST"
# GitHub organizasyon adı tanımlanır.


GITHUB_REPOSITORY = "MOL-FET_regression"
# GitHub repo adı tanımlanır.


GITHUB_BRANCH = "main"
# Verinin okunacağı GitHub branch adı tanımlanır.


TARGET_SPECIFIC_INPUT_FILENAME = (
    f"{TARGET_ID}_Mordred2D_model_ready.csv"
)
# Workshop notebook 02 tarafından üretilebilecek target'a özel feature store adı tanımlanır.


REPOSITORY_FEATURE_FILENAME = "mordred_2d_features.csv"
# Mevcut GitHub reposunda gerçekten bulunan Mordred feature store dosya adı tanımlanır.


DRIVE_ROOT = Path(
    "/content/drive/MyDrive/MOL_FET_regression_workshop"
)
# Bütün model ve rapor çıktılarının kaydedileceği Drive klasörü tanımlanır.


DRIVE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)
# Drive klasörü yoksa oluşturulur.


RANDOM_STATE = 42
# Train/test ayrımı ve modeller için tekrar üretilebilir seed belirlenir.


TEST_SIZE = 0.20
# Test seti oranı yüzde 20 olarak belirlenir.


CV_FOLDS = 5
# Final model beş katlı çapraz doğrulama ile değerlendirilir.


REUSE_VALID_LAZYPREDICT_CACHE = True
# Aynı veri ve ayarlarla daha önce oluşturulmuş LazyPredict sonucu varsa yeniden tarama yapılmaz.


MAX_FEATURE_MISSING_RATIO = 0.20
# Bir descriptorın kabul edilen maksimum eksik hücre oranı belirlenir.


REQUEST_TIMEOUT = 180
# GitHub istekleri için saniye cinsinden zaman aşımı belirlenir.


print("TARGET_ID:", TARGET_ID)
print("Target'a özel aday:", TARGET_SPECIFIC_INPUT_FILENAME)
print("Repo feature store adayı:", REPOSITORY_FEATURE_FILENAME)
print("Drive çıktı klasörü:", DRIVE_ROOT)
print("✅ CHECKPOINT 03.4: Çalışma ayarları hazır.")

## Hücre 5 — GitHub repo ve aday dosya yolları

In [ ]:
PREFERRED_GITHUB_PATHS = [
    f"data/{TARGET_SPECIFIC_INPUT_FILENAME}",
    # Target'a özel dosyanın tercih edilen data/ klasörü yolu.

    TARGET_SPECIFIC_INPUT_FILENAME,
    # Target'a özel dosya repo köküne yüklenmişse kullanılacak yol.

    (
        "molfet_regression_outputs/"
        "01_mordred_feature_store/"
        "mordred_2d_features.csv"
    ),
    # Yüklenen gerçek GitHub reposunda bulunan mevcut Mordred feature store yolu.

    REPOSITORY_FEATURE_FILENAME,
    # Aynı dosya daha sonra repo köküne taşınırsa kullanılacak yol.
]
# Gerçek repo yapısı dahil olmak üzere denenecek yollar öncelik sırasıyla tanımlanır.


GITHUB_API_TREE_URL = (
    f"https://api.github.com/repos/{GITHUB_OWNER}/"
    f"{GITHUB_REPOSITORY}/git/trees/{GITHUB_BRANCH}?recursive=1"
)
# Repo içindeki bütün dosyaları recursive listeleyen GitHub API adresi oluşturulur.


print("GitHub aday yolları:")
for candidate_path in PREFERRED_GITHUB_PATHS:
    print(" -", candidate_path)


print("✅ CHECKPOINT 03.5: GitHub aday yolları hazır.")

## Hücre 6 — GitHub RAW URL fonksiyonu

In [ ]:
def build_raw_url(relative_path):
    """Repo içindeki göreli yolu GitHub RAW URL adresine dönüştürür."""

    clean_path = str(relative_path).lstrip("/")
    # Yolun başındaki olası eğik çizgi kaldırılır.

    return (
        f"https://raw.githubusercontent.com/"
        f"{GITHUB_OWNER}/{GITHUB_REPOSITORY}/"
        f"{GITHUB_BRANCH}/{clean_path}"
    )
    # Tam GitHub RAW adresi döndürülür.


print("✅ CHECKPOINT 03.6: RAW URL fonksiyonu hazır.")

## Hücre 7 — GitHub repo ağacı fonksiyonu

In [ ]:
def get_repository_file_paths():
    """GitHub API üzerinden repo içindeki bütün dosya yollarını döndürür."""

    response = requests.get(
        GITHUB_API_TREE_URL,
        timeout=REQUEST_TIMEOUT,
        headers={
            "Accept": "application/vnd.github+json",
        },
    )
    # Recursive GitHub repo ağacı istenir.

    response.raise_for_status()
    # HTTP hata kodlarında exception oluşturulur.

    payload = response.json()
    # JSON yanıtı Python sözlüğüne dönüştürülür.

    file_paths = [
        entry.get("path")
        for entry in payload.get("tree", [])
        if entry.get("type") == "blob" and entry.get("path")
    ]
    # Yalnızca gerçek dosya girişlerinin göreli yolları seçilir.

    if not file_paths:
        # Dosya listesi boşsa kontrol edilir.

        raise RuntimeError(
            "GitHub API repo ağacında dosya döndürmedi."
        )
        # Boş repo ağacıyla devam edilmez.

    return file_paths
    # Repo içindeki bütün dosya yolları döndürülür.


print("✅ CHECKPOINT 03.7: GitHub repo ağacı fonksiyonu hazır.")

## Hücre 8 — Mordred CSV yolunu otomatik bulma fonksiyonu

In [ ]:
def resolve_mordred_csv_path():
    """Repo içinde modellemeye uygun Mordred feature store CSV yolunu bulur."""

    try:
        # Öncelikle GitHub API üzerinden repo ağacı okunur.

        repository_files = get_repository_file_paths()
        # Repo içindeki bütün dosya yolları alınır.

        repository_file_set = set(repository_files)
        # Tam yol kontrolünü hızlandırmak için küme oluşturulur.

        for preferred_path in PREFERRED_GITHUB_PATHS:
            # Gerçek repo yolu dahil öncelikli yollar sırayla kontrol edilir.

            if preferred_path in repository_file_set:
                # Dosya repoda varsa seçilir.

                print("GitHub API ile bulunan feature store:", preferred_path)
                # Bulunan yol kullanıcıya gösterilir.

                return preferred_path
                # İlk bulunan öncelikli yol döndürülür.

        accepted_filenames = {
            TARGET_SPECIFIC_INPUT_FILENAME,
            REPOSITORY_FEATURE_FILENAME,
        }
        # Desteklenen tam dosya adları tanımlanır.

        exact_name_matches = [
            path
            for path in repository_files
            if Path(path).name in accepted_filenames
        ]
        # Dosya başka bir alt klasördeyse desteklenen adlarla recursive aranır.

        if exact_name_matches:
            # En az bir tam isim eşleşmesi bulunduysa kontrol edilir.

            exact_name_matches = sorted(
                exact_name_matches,
                key=lambda path: (
                    Path(path).name != TARGET_SPECIFIC_INPUT_FILENAME,
                    "01_mordred_feature_store" not in path,
                    len(Path(path).parts),
                    path,
                ),
            )
            # Target'a özel dosya ve mevcut feature store klasörü önceliklendirilir.

            selected_path = exact_name_matches[0]
            # En uygun yol seçilir.

            print("Recursive arama ile bulunan feature store:", selected_path)
            # Seçilen yol kullanıcıya gösterilir.

            return selected_path
            # Seçilen yol döndürülür.

        mordred_feature_matches = [
            path
            for path in repository_files
            if (
                str(path).lower().endswith(".csv")
                and "mordred" in Path(path).name.lower()
                and (
                    "feature" in Path(path).name.lower()
                    or "model_ready" in Path(path).name.lower()
                )
                and "missingness" not in Path(path).name.lower()
                and "index" not in Path(path).name.lower()
                and "prediction" not in Path(path).name.lower()
                and "importance" not in Path(path).name.lower()
                and "metrics" not in Path(path).name.lower()
            )
        ]
        # İsim biraz değişmişse gerçek descriptor matrisi olabilecek CSV'ler aranır.

        if mordred_feature_matches:
            # Anahtar kelime eşleşmesi bulunduysa kontrol edilir.

            mordred_feature_matches = sorted(
                mordred_feature_matches,
                key=lambda path: (
                    "01_mordred_feature_store" not in path,
                    len(Path(path).parts),
                    path,
                ),
            )
            # Feature-store klasöründeki daha kısa yol önceliklendirilir.

            selected_path = mordred_feature_matches[0]
            # En uygun aday seçilir.

            print("Anahtar kelime aramasıyla bulunan feature store:", selected_path)
            # Seçilen yol kullanıcıya gösterilir.

            return selected_path
            # Seçilen yol döndürülür.

        available_csv_files = sorted(
            path
            for path in repository_files
            if str(path).lower().endswith(".csv")
        )
        # Tanı amacıyla repodaki CSV dosyaları listelenir.

        raise FileNotFoundError(
            "Modellemeye uygun Mordred feature store bulunamadı.\n"
            "Repo içindeki CSV dosyaları:\n- "
            + "\n- ".join(available_csv_files[:50])
        )
        # Uygun feature store yoksa açıklayıcı hata oluşturulur.

    except requests.RequestException as api_error:
        # GitHub API erişimi başarısızsa hata yakalanır.

        print("GitHub API erişimi başarısız:", api_error)
        # API hatası bilgi amaçlı gösterilir.

        return None
        # Öncelikli RAW yollarının doğrudan denenmesi sağlanır.


print("✅ CHECKPOINT 03.8: Mordred CSV çözümleme fonksiyonu hazır.")

## Hücre 9 — GitHub CSV indirme fonksiyonu

In [ ]:
def looks_like_mordred_feature_store(dataframe):
    """CSV'nin target, SMILES ve Mordred descriptor sütunlarını içerdiğini kontrol eder."""

    target_candidates = {
        "Target",
        "target",
        "pStandard",
        "pIC50",
        "y",
    }
    # Desteklenen target sütun adları tanımlanır.

    smiles_candidates = {
        "SMILES",
        "smiles",
        "canonical_smiles",
        "parent_smiles",
    }
    # Desteklenen SMILES sütun adları tanımlanır.

    has_target = any(
        column in dataframe.columns
        for column in target_candidates
    )
    # En az bir target sütunu olup olmadığı kontrol edilir.

    has_smiles = any(
        column in dataframe.columns
        for column in smiles_candidates
    )
    # En az bir SMILES sütunu olup olmadığı kontrol edilir.

    mordred_columns = [
        column
        for column in dataframe.columns
        if str(column).startswith("Mordred_")
    ]
    # Mevcut repo formatındaki Mordred_ önekli descriptorlar belirlenir.

    numeric_candidate_count = sum(
        pd.api.types.is_numeric_dtype(dataframe[column])
        for column in dataframe.columns
        if column not in target_candidates
    )
    # Target dışında sayısal olabilecek sütun sayısı hesaplanır.

    return (
        has_target
        and has_smiles
        and (
            len(mordred_columns) >= 10
            or numeric_candidate_count >= 10
        )
    )
    # Target, SMILES ve yeterli descriptor varsa uygun feature store kabul edilir.


def download_csv_candidate(relative_path):
    """Bir GitHub yolundaki CSV'yi indirir ve feature store yapısını doğrular."""

    raw_url = build_raw_url(relative_path)
    # Göreli yol tam RAW URL adresine dönüştürülür.

    print("Denenen GitHub RAW URL:", raw_url)
    # Test edilen bağlantı gösterilir.

    response = requests.get(
        raw_url,
        timeout=REQUEST_TIMEOUT,
    )
    # GitHub RAW bağlantısına istek gönderilir.

    if response.status_code == 404:
        # Dosya belirtilen konumda yoksa kontrol edilir.

        print("  → 404: Bu konumda dosya yok.")
        # 404 sonucu gösterilir.

        return None, raw_url
        # Pipeline durmadan sonraki aday yol denenir.

    response.raise_for_status()
    # 404 dışındaki HTTP hatalarında exception oluşturulur.

    if not response.content:
        # İndirilen içerik boşsa kontrol edilir.

        print("  → Dosya boş.")
        # Boş dosya bilgisi gösterilir.

        return None, raw_url
        # Boş dosya kullanılmaz.

    try:
        # CSV ayrıştırma hatasını yakalamak için try bloğu kullanılır.

        dataframe = pd.read_csv(
            BytesIO(response.content),
            low_memory=False,
        )
        # İndirilen içerik pandas DataFrame'e dönüştürülür.

    except Exception as csv_error:
        # CSV okuma hatası yakalanır.

        print("  → CSV okuma hatası:", csv_error)
        # Ayrıştırma hatası gösterilir.

        return None, raw_url
        # Geçersiz içerik kullanılmaz.

    if dataframe.empty:
        # DataFrame boşsa kontrol edilir.

        print("  → CSV boş.")
        # Boş tablo bilgisi gösterilir.

        return None, raw_url
        # Boş DataFrame kullanılmaz.

    if not looks_like_mordred_feature_store(dataframe):
        # Dosyanın gerçek descriptor matrisi olup olmadığı kontrol edilir.

        print(
            "  → CSV feature store yapısına uymuyor. "
            "İlk sütunlar:",
            list(dataframe.columns[:12]),
        )
        # feature_store_index veya rapor CSV'lerinin yanlışlıkla seçilmesi engellenir.

        return None, raw_url
        # Uygun olmayan CSV atlanır.

    print("  → Modellemeye uygun Mordred feature store bulundu.")
    # Başarılı okuma bilgisi gösterilir.

    return dataframe, raw_url
    # DataFrame ve kullanılan RAW URL döndürülür.


print("✅ CHECKPOINT 03.9: GitHub CSV indirme ve doğrulama fonksiyonları hazır.")

## Hücre 10 — GitHub Mordred CSV'nin okunması

In [ ]:
resolved_github_path = resolve_mordred_csv_path()
# GitHub repo ağacından en uygun Mordred CSV yolu bulunur.


github_paths_to_try = []
# Denenecek repo yollarını tutacak liste oluşturulur.


if resolved_github_path is not None:
    # API bir yol bulduysa kontrol edilir.

    github_paths_to_try.append(resolved_github_path)
    # Bulunan yol ilk sıraya eklenir.


for preferred_path in PREFERRED_GITHUB_PATHS:
    # Öncelikli yollar sırayla dolaşılır.

    if preferred_path not in github_paths_to_try:
        # Aynı yol daha önce eklenmediyse kontrol edilir.

        github_paths_to_try.append(preferred_path)
        # Aday yol deneme listesine eklenir.


df = None
# Başlangıçta henüz veri okunmadığı belirtilir.


USED_GITHUB_PATH = None
# Kullanılan repo içi yol daha sonra kaydetmek için boş tanımlanır.


USED_GITHUB_URL = None
# Kullanılan RAW URL daha sonra kaydetmek için boş tanımlanır.


for candidate_path in github_paths_to_try:
    # Bütün aday yollar sırayla denenir.

    candidate_df, candidate_url = download_csv_candidate(
        candidate_path
    )
    # Aday CSV indirilir.

    if candidate_df is not None:
        # Geçerli DataFrame bulunduysa kontrol edilir.

        df = candidate_df
        # Geçerli veri ana DataFrame olarak atanır.

        USED_GITHUB_PATH = candidate_path
        # Başarılı repo yolu saklanır.

        USED_GITHUB_URL = candidate_url
        # Başarılı RAW URL saklanır.

        break
        # İlk başarılı dosyada döngü sonlandırılır.


if df is None:
    # Hiçbir adaydan veri okunamadıysa kontrol edilir.

    raise FileNotFoundError(
        "GitHub reposundan modellemeye uygun Mordred feature store okunamadı."
    )
    # Açıklayıcı hata oluşturulur.


print("Kullanılan GitHub yolu:", USED_GITHUB_PATH)
print("Kullanılan GitHub RAW URL:", USED_GITHUB_URL)
print("Ham feature store boyutu:", df.shape)
display(df.head(10))
print("✅ CHECKPOINT 03.10: Mordred feature store başarıyla okundu.")

## Hücre 11 — Target ve metadata sütun adayları

In [ ]:
TARGET_COLUMN_CANDIDATES = [
    "Target",
    # Mevcut GitHub feature store dosyasındaki target sütunu.

    "target",
    # Workshop notebook 02'nin ürettiği standart target sütunu.

    "pStandard",
    "pStandard_mean",
    "pIC50",
    "y",
]
# Regression target sütunu için olası adlar tanımlanır.


METADATA_CANDIDATES = [
    "MoleculeID",
    # Mevcut GitHub feature store dosyasındaki molekül kimliği.

    "SMILES",
    # Mevcut GitHub feature store dosyasındaki SMILES sütunu.

    "Target",
    # Mevcut GitHub feature store dosyasındaki target sütunu.

    "molecule_id",
    "canonical_smiles",
    "parent_smiles",
    "target",
    "pStandard",
    "pStandard_mean",
    "pStandard_std",
    "target_mean",
    "target_std",
    "source_rows",
    "target_source_column",
    "target_chembl_id",
    "parent_chembl_id",
    "MolWt",
    "MolLogP",
    "HBD",
    "HBA",
]
# Model featureı olarak kullanılmaması gereken açıklayıcı sütunlar tanımlanır.


print("✅ CHECKPOINT 03.11: Target ve metadata adayları hazır.")

## Hücre 12 — Target sütununu seçme fonksiyonu

In [ ]:
def detect_column(dataframe, candidates, label):
    """Aday sütunlardan DataFrame içinde bulunan ilk sütunu döndürür."""

    for column in candidates:
        # Aday sütunlar sırayla kontrol edilir.

        if column in dataframe.columns:
            # Sütun veri setinde bulunursa seçilir.

            print(f"{label} sütunu seçildi: {column}")
            # Seçilen sütun adı gösterilir.

            return column
            # İlk uygun sütun döndürülür.

    raise ValueError(
        f"{label} sütunu bulunamadı. Adaylar: {candidates}"
    )
    # Hiçbir aday bulunamazsa açıklayıcı hata oluşturulur.


print("✅ CHECKPOINT 03.12: Target sütunu seçme fonksiyonu hazır.")

## Hücre 13 — Target, metadata ve feature sütunlarının ayrılması

In [ ]:
target_column = detect_column(
    df,
    TARGET_COLUMN_CANDIDATES,
    "Regression target",
)
# Regression target sütunu otomatik belirlenir.


metadata_columns = [
    column
    for column in METADATA_CANDIDATES
    if column in df.columns
]
# Veri içinde gerçekten bulunan metadata sütunları seçilir.


mordred_prefixed_columns = [
    column
    for column in df.columns
    if str(column).startswith("Mordred_")
]
# Mevcut GitHub formatında Mordred_ önekli descriptor sütunları belirlenir.


if mordred_prefixed_columns:
    # Mordred_ önekli descriptorlar varsa kontrol edilir.

    feature_columns = mordred_prefixed_columns
    # Yalnızca gerçek Mordred descriptorları modele alınır.

else:
    # Target'a özel workshop CSV'sinde Mordred_ öneki yoksa alternatif kola geçilir.

    feature_columns = [
        column
        for column in df.columns
        if column not in metadata_columns
    ]
    # Metadata dışındaki sütunlar model featureı olarak seçilir.


if not feature_columns:
    # Feature sütunu bulunup bulunmadığı kontrol edilir.

    raise ValueError(
        "Modelleme için Mordred descriptor sütunu bulunamadı."
    )
    # Feature yoksa pipeline durdurulur.


print("Target sütunu:", target_column)
print("Metadata sütunu sayısı:", len(metadata_columns))
print("Feature sütunu sayısı:", len(feature_columns))
print("İlk featurelar:", feature_columns[:10])
print("✅ CHECKPOINT 03.13: Sütun grupları ayrıldı.")

## Hücre 14 — Feature matrisi ve target dizisinin hazırlanması

In [ ]:
X_raw = (
    df[feature_columns]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
)
# Mordred descriptor sütunları sayısala dönüştürülür; hata nesneleri NaN olur.


X_raw = X_raw.replace(
    [
        np.inf,
        -np.inf,
    ],
    np.nan,
)
# Pozitif ve negatif sonsuz değerler güvenli biçimde NaN'a çevrilir.


y = pd.to_numeric(
    df[target_column],
    errors="coerce",
)
# Regression target sütunu sayısal vektöre dönüştürülür.


print("Ham X boyutu:", X_raw.shape)
print("Ham y boyutu:", y.shape)
print(
    "Ham feature matrisindeki NaN/sonsuz hücre:",
    int(X_raw.isna().sum().sum()),
)
print("✅ CHECKPOINT 03.14: Ham feature matrisi ve target dizisi hazırlandı.")

## Hücre 15 — Feature ve target kalite kontrolü

In [ ]:
valid_target_mask = y.notna()
# Sayısal target değeri bulunan satırlar belirlenir.


removed_target_rows = int((~valid_target_mask).sum())
# Eksik veya geçersiz target nedeniyle çıkarılacak satır sayısı hesaplanır.


df = df.loc[
    valid_target_mask
].reset_index(drop=True)
# Geçerli target satırları ana metadata tablosunda tutulur.


X_raw = X_raw.loc[
    valid_target_mask
].reset_index(drop=True)
# Feature matrisi aynı satır maskesiyle hizalanır.


y = y.loc[
    valid_target_mask
].reset_index(drop=True).astype(float)
# Target vektörü aynı satır maskesiyle hizalanır.


all_missing_columns = X_raw.columns[
    X_raw.isna().all(axis=0)
].tolist()
# Bütün moleküllerde eksik olan descriptorlar belirlenir.


X_filtered = X_raw.drop(
    columns=all_missing_columns
).copy()
# Tamamen boş descriptorlar feature matrisinden çıkarılır.


missing_ratio = X_filtered.isna().mean()
# Her descriptor için eksik değer oranı hesaplanır.


high_missing_columns = missing_ratio[
    missing_ratio > MAX_FEATURE_MISSING_RATIO
].index.tolist()
# Eksik oranı belirlenen eşiği aşan descriptorlar belirlenir.


X_filtered = X_filtered.drop(
    columns=high_missing_columns
).copy()
# Yüksek eksik oranlı descriptorlar çıkarılır.


constant_columns = [
    column
    for column in X_filtered.columns
    if X_filtered[column].nunique(dropna=True) <= 1
]
# Eksik olmayan değerleri tek bir sabit değerden oluşan descriptorlar belirlenir.


X_filtered = X_filtered.drop(
    columns=constant_columns
).copy()
# Sabit descriptorlar model matrisinden çıkarılır.


feature_columns = X_filtered.columns.tolist()
# Final feature sırası temizlenmiş matris üzerinden güncellenir.


X = X_filtered.astype(np.float32)
# Temizlenmiş fakat henüz imputasyon yapılmamış matris float32 biçimine dönüştürülür.


if y.isna().any():
    raise AssertionError(
        "Target temizliği sonrasında eksik target kaldı."
    )


if X.empty:
    raise ValueError(
        "Feature temizliğinden sonra descriptor kalmadı."
    )


if len(X) < 10:
    raise ValueError(
        "Regression modellemesi için en az 10 molekül gereklidir."
    )


print("Eksik target nedeniyle çıkarılan satır:", removed_target_rows)
print("Tamamen boş çıkarılan feature:", len(all_missing_columns))
print("Yüksek eksik oranıyla çıkarılan feature:", len(high_missing_columns))
print("Sabit çıkarılan feature:", len(constant_columns))
print("Modellemeye kalan molekül:", len(df))
print("Modellemeye kalan feature:", len(feature_columns))
print("Kalan eksik hücre:", int(X.isna().sum().sum()))
print("Target aralığı:", float(y.min()), "—", float(y.max()))
print("✅ CHECKPOINT 03.15: Feature ve target kalite kontrolü tamamlandı.")

## Hücre 16 — Train/test indekslerinin oluşturulması

In [ ]:
all_indices = np.arange(
    len(df)
)
# Bütün satır indeksleri NumPy dizisi olarak oluşturulur.


train_indices, test_indices = train_test_split(
    all_indices,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True,
)
# Veri yüzde 80 eğitim ve yüzde 20 test olarak bir kez ayrılır.


if not set(train_indices).isdisjoint(
    set(test_indices)
):
    # Train ve test indekslerinin çakışıp çakışmadığı kontrol edilir.

    raise AssertionError(
        "Train ve test indeksleri çakışıyor."
    )
    # Çakışma varsa pipeline durdurulur.


if len(train_indices) + len(test_indices) != len(df):
    # Bütün satırların bir split'e atanıp atanmadığı kontrol edilir.

    raise AssertionError(
        "Train ve test indeksleri bütün satırları kapsamıyor."
    )
    # Eksik veya fazla indeks varsa pipeline durdurulur.


print("Train satırı:", len(train_indices))
print("Test satırı:", len(test_indices))
print("✅ CHECKPOINT 03.16: Train/test indeksleri bir kez oluşturuldu.")

## Hücre 17 — Train ve test alt kümelerinin oluşturulması

In [ ]:
X_train_raw = X.iloc[
    train_indices
].copy()
# Eğitim feature matrisi imputasyon öncesi oluşturulur.


X_test_raw = X.iloc[
    test_indices
].copy()
# Test feature matrisi imputasyon öncesi oluşturulur.


y_train = y.iloc[
    train_indices
].copy()
# Eğitim target vektörü oluşturulur.


y_test = y.iloc[
    test_indices
].copy()
# Test target vektörü oluşturulur.


feature_imputer = SimpleImputer(
    strategy="median",
)
# Eksik descriptorları eğitim medyanıyla dolduracak imputer oluşturulur.


X_train = pd.DataFrame(
    feature_imputer.fit_transform(X_train_raw),
    columns=feature_columns,
    index=X_train_raw.index,
).astype(np.float32)
# Imputer yalnızca eğitim setinde fit edilir ve eğitim matrisi dönüştürülür.


X_test = pd.DataFrame(
    feature_imputer.transform(X_test_raw),
    columns=feature_columns,
    index=X_test_raw.index,
).astype(np.float32)
# Eğitimde öğrenilen medyanlar değiştirilmeden test setine uygulanır.


if not np.isfinite(X_train.to_numpy()).all():
    raise AssertionError(
        "İmputasyon sonrasında eğitim matrisinde geçersiz değer kaldı."
    )


if not np.isfinite(X_test.to_numpy()).all():
    raise AssertionError(
        "İmputasyon sonrasında test matrisinde geçersiz değer kaldı."
    )


print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)
print("Train eksik hücre:", int(X_train.isna().sum().sum()))
print("Test eksik hücre:", int(X_test.isna().sum().sum()))
print("✅ CHECKPOINT 03.17: Train tabanlı median imputasyon tamamlandı.")

## Hücre 18 — Regression metrik fonksiyonu

In [ ]:
def regression_metrics(y_true, y_pred):
    """R2, RMSE ve MAE metriklerini sözlük olarak döndürür."""

    r2_value = float(
        r2_score(
            y_true,
            y_pred,
        )
    )
    # Açıklanan varyans oranı hesaplanır.

    rmse_value = float(
        np.sqrt(
            mean_squared_error(
                y_true,
                y_pred,
            )
        )
    )
    # Kök ortalama kare hata hesaplanır.

    mae_value = float(
        mean_absolute_error(
            y_true,
            y_pred,
        )
    )
    # Ortalama mutlak hata hesaplanır.

    return {
        "R2": r2_value,
        "RMSE": rmse_value,
        "MAE": mae_value,
    }
    # Üç regression metriği sözlük olarak döndürülür.


print("✅ CHECKPOINT 03.18: Regression metrik fonksiyonu hazır.")

## Hücre 19 — Final ağaç modeli üretme fonksiyonu

In [ ]:
MODEL_PARAMETER_MAP = {
    "ExtraTreesRegressor": {
        "n_estimators": 500,
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
    },
    # Extra Trees workshop parametreleri tanımlanır.

    "RandomForestRegressor": {
        "n_estimators": 500,
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
    },
    # Random Forest workshop parametreleri tanımlanır.

    "GradientBoostingRegressor": {
        "random_state": RANDOM_STATE,
    },
    # Gradient Boosting workshop parametreleri tanımlanır.
}
# SHAP uyumlu final model adayları ve parametreleri tek yerde tanımlanır.


MODEL_BUILDERS = {
    "ExtraTreesRegressor": ExtraTreesRegressor,
    "RandomForestRegressor": RandomForestRegressor,
    "GradientBoostingRegressor": GradientBoostingRegressor,
}
# Model adları sklearn model sınıflarıyla eşlenir.


def build_model(model_name):
    """Model adına göre yeni ve temiz sklearn modeli oluşturur."""

    if model_name not in MODEL_BUILDERS:
        # Model adının desteklenen listede olup olmadığı kontrol edilir.

        raise ValueError(
            f"Desteklenmeyen final model: {model_name}"
        )
        # Desteklenmeyen modelde pipeline durdurulur.

    model_class = MODEL_BUILDERS[model_name]
    # Model adına karşılık gelen sklearn sınıfı alınır.

    model_parameters = MODEL_PARAMETER_MAP[model_name]
    # Aynı modele ait parametreler alınır.

    return model_class(
        **model_parameters
    )
    # Yeni ve henüz fit edilmemiş model nesnesi döndürülür.


print("✅ CHECKPOINT 03.19: Final model üretme fonksiyonu hazır.")

## Hücre 20 — LazyPredict cache dosya yolları

In [ ]:
lazy_results_filename = (
    f"{TARGET_ID}_LazyPredict_results.csv"
)
# LazyPredict model karşılaştırma CSV adı oluşturulur.


lazy_predictions_filename = (
    f"{TARGET_ID}_LazyPredict_test_predictions.csv"
)
# LazyPredict test tahmin CSV adı oluşturulur.


lazy_cache_metadata_filename = (
    f"{TARGET_ID}_LazyPredict_cache_metadata.json"
)
# LazyPredict cache doğrulama metadata dosya adı oluşturulur.


lazy_results_path = (
    DRIVE_ROOT / lazy_results_filename
)
# LazyPredict sonuç tablosu Drive yolu oluşturulur.


lazy_predictions_path = (
    DRIVE_ROOT / lazy_predictions_filename
)
# LazyPredict tahmin tablosu Drive yolu oluşturulur.


lazy_cache_metadata_path = (
    DRIVE_ROOT / lazy_cache_metadata_filename
)
# LazyPredict cache metadata Drive yolu oluşturulur.


print("✅ CHECKPOINT 03.20: LazyPredict cache yolları hazır.")

## Hücre 21 — Veri ve ayar imzasının oluşturulması

In [ ]:
signature_payload = {
    "target_id": TARGET_ID,
    "github_path": USED_GITHUB_PATH,
    "github_url": USED_GITHUB_URL,
    "rows": int(len(df)),
    "features": int(len(feature_columns)),
    "feature_names": feature_columns,
    "target_column": target_column,
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "train_indices": train_indices.tolist(),
    "test_indices": test_indices.tolist(),
}
# Cache'in hangi veri ve ayarlara ait olduğunu tanımlayan içerik hazırlanır.


signature_text = json.dumps(
    signature_payload,
    sort_keys=True,
    ensure_ascii=False,
)
# İmza içeriği değişmez sıralı JSON metnine dönüştürülür.


PIPELINE_SIGNATURE = hashlib.sha256(
    signature_text.encode("utf-8")
).hexdigest()
# Veri ve ayar içeriğinden SHA-256 pipeline imzası oluşturulur.


print("Pipeline imzası:", PIPELINE_SIGNATURE[:16], "...")
print("✅ CHECKPOINT 03.21: Veri ve ayar imzası oluşturuldu.")

## Hücre 22 — LazyPredict cache geçerlilik kontrolü

In [ ]:
lazy_cache_is_valid = False
# Başlangıçta cache geçersiz kabul edilir.


if REUSE_VALID_LAZYPREDICT_CACHE:
    # Cache kullanımına izin verildiyse kontrol yapılır.

    required_cache_paths = [
        lazy_results_path,
        lazy_predictions_path,
        lazy_cache_metadata_path,
    ]
    # Geçerli cache için gerekli üç dosya tanımlanır.

    if all(
        path.exists()
        for path in required_cache_paths
    ):
        # Bütün cache dosyaları mevcutsa kontrol edilir.

        try:
            # Cache metadata okuma hatalarını yakalamak için try bloğu başlatılır.

            with open(
                lazy_cache_metadata_path,
                "r",
                encoding="utf-8",
            ) as file:
                # Cache metadata dosyası okuma modunda açılır.

                lazy_cache_metadata = json.load(file)
                # JSON metadata Python sözlüğüne dönüştürülür.

            lazy_cache_is_valid = (
                lazy_cache_metadata.get("pipeline_signature")
                == PIPELINE_SIGNATURE
            )
            # Kaydedilmiş imza güncel veri ve ayar imzasıyla karşılaştırılır.

        except Exception as cache_error:
            # Cache dosyası bozuk veya okunamazsa hata yakalanır.

            print("Cache metadata okunamadı:", cache_error)
            # Hata bilgi amaçlı gösterilir.

            lazy_cache_is_valid = False
            # Bozuk cache kullanılmaz.


print("Geçerli LazyPredict cache:", lazy_cache_is_valid)
print("✅ CHECKPOINT 03.22: LazyPredict cache kontrolü tamamlandı.")

## Hücre 23 — LazyPredict çalıştırma veya cache yükleme

In [ ]:
if lazy_cache_is_valid:
    # Aynı veri ve ayarlara ait geçerli cache varsa kontrol edilir.

    lazy_models = pd.read_csv(
        lazy_results_path,
        low_memory=False,
    )
    # Daha önce kaydedilmiş LazyPredict sonuç tablosu okunur.

    if "Model" not in lazy_models.columns:
        # Model adı sütununun bulunup bulunmadığı kontrol edilir.

        raise ValueError(
            "LazyPredict cache sonuçlarında Model sütunu yok."
        )
        # Hatalı cache ile devam edilmez.

    lazy_models = lazy_models.set_index(
        "Model"
    )
    # Model adı tekrar tablo indeksi yapılır.

    lazy_predictions = pd.read_csv(
        lazy_predictions_path,
        index_col=0,
        low_memory=False,
    )
    # Daha önce kaydedilmiş LazyPredict tahmin tablosu okunur.

    print("Geçerli cache kullanıldı; LazyPredict yeniden çalıştırılmadı.")
    # Pahalı model taramasının atlandığı bildirilir.

else:
    # Geçerli cache yoksa LazyPredict yalnızca bir kez çalıştırılır.

    lazy_regressor = LazyRegressor(
        verbose=0,
        ignore_warnings=True,
        custom_metric=None,
        predictions=True,
        random_state=RANDOM_STATE,
    )
    # LazyPredict regression tarayıcısı oluşturulur.

    lazy_models, lazy_predictions = lazy_regressor.fit(
        X_train,
        X_test,
        y_train,
        y_test,
    )
    # Temel regression algoritmaları aynı train/test split üzerinde karşılaştırılır.

    if lazy_models.empty:
        # Sonuç tablosunun boş olup olmadığı kontrol edilir.

        raise RuntimeError(
            "LazyPredict hiçbir model sonucu üretmedi."
        )
        # Sonuç üretilemezse pipeline durdurulur.

    lazy_models.reset_index().rename(
        columns={
            "index": "Model",
        }
    ).to_csv(
        lazy_results_path,
        index=False,
    )
    # LazyPredict karşılaştırma tablosu Drive'a kaydedilir.

    lazy_predictions.to_csv(
        lazy_predictions_path,
        index=True,
    )
    # LazyPredict test tahminleri Drive'a kaydedilir.

    with open(
        lazy_cache_metadata_path,
        "w",
        encoding="utf-8",
    ) as file:
        # Cache metadata dosyası yazma modunda açılır.

        json.dump(
            {
                "pipeline_signature": PIPELINE_SIGNATURE,
                **signature_payload,
            },
            file,
            ensure_ascii=False,
            indent=2,
        )
        # Güncel veri ve ayar imzası Drive'a kaydedilir.

    print("LazyPredict çalıştırıldı ve yeni cache oluşturuldu.")
    # Yeni tarama yapıldığı bildirilir.


display(
    lazy_models.head(20)
)
# En iyi 20 model gösterilir.


print("✅ CHECKPOINT 03.23: LazyPredict sonuçları hazır.")

## Hücre 24 — Final model adının seçilmesi

In [ ]:
selected_model_name = next(
    (
        str(model_name)
        for model_name in lazy_models.index
        if str(model_name) in MODEL_PARAMETER_MAP
    ),
    "ExtraTreesRegressor",
)
# LazyPredict sıralamasındaki ilk desteklenen SHAP uyumlu ağaç modeli seçilir.


selected_model_params = MODEL_PARAMETER_MAP[
    selected_model_name
].copy()
# Seçilen modele ait parametrelerin kopyası alınır.


print("Seçilen final model:", selected_model_name)
print("Model parametreleri:", selected_model_params)
print("✅ CHECKPOINT 03.24: Final model seçildi.")

## Hücre 25 — Final modelin eğitilmesi

In [ ]:
final_model = build_model(
    selected_model_name
)
# Seçilen model için yeni ve temiz sklearn modeli oluşturulur.


final_model.fit(
    X_train,
    y_train,
)
# Final model yalnızca eğitim verisinde bir kez fit edilir.


print("✅ CHECKPOINT 03.25: Final model eğitim verisinde fit edildi.")

## Hücre 26 — Train ve test tahminlerinin oluşturulması

In [ ]:
train_predictions = final_model.predict(
    X_train
)
# Eğitim seti tahminleri üretilir.


test_predictions = final_model.predict(
    X_test
)
# Test seti tahminleri üretilir.


print("Train tahmin sayısı:", len(train_predictions))
print("Test tahmin sayısı:", len(test_predictions))
print("✅ CHECKPOINT 03.26: Train ve test tahminleri hazır.")

## Hücre 27 — Holdout metriklerinin hesaplanması

In [ ]:
train_metrics = regression_metrics(
    y_train,
    train_predictions,
)
# Eğitim R2, RMSE ve MAE değerleri hesaplanır.


test_metrics = regression_metrics(
    y_test,
    test_predictions,
)
# Test R2, RMSE ve MAE değerleri hesaplanır.


metrics_table = pd.DataFrame(
    [
        train_metrics,
        test_metrics,
    ],
    index=[
        "Train",
        "Test",
    ],
)
# Train ve test metrikleri karşılaştırmalı tabloya dönüştürülür.


display(metrics_table)
# Holdout metrikleri gösterilir.


print("✅ CHECKPOINT 03.27: Holdout metrikleri hesaplandı.")

## Hücre 28 — 5-fold CV bölücüsünün hazırlanması

In [ ]:
cv_splitter = KFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)
# Tekrar üretilebilir beş katlı KFold bölücüsü oluşturulur.


cv_model = build_model(
    selected_model_name
)
# Çapraz doğrulama için final modelden bağımsız temiz bir model nesnesi oluşturulur.


print("✅ CHECKPOINT 03.28: CV bölücüsü ve temiz model hazır.")

## Hücre 29 — 5-fold çapraz doğrulamanın çalıştırılması

In [ ]:
cv_results = cross_validate(
    cv_model,
    X_train,
    y_train,
    cv=cv_splitter,
    scoring={
        "R2": "r2",
        "RMSE": "neg_root_mean_squared_error",
        "MAE": "neg_mean_absolute_error",
    },
    n_jobs=-1,
)
# Final model yalnızca eğitim seti içinde beş katlı çapraz doğrulanır.


print("✅ CHECKPOINT 03.29: 5-fold çapraz doğrulama tamamlandı.")

## Hücre 30 — CV özet tablosunun hazırlanması

In [ ]:
cv_summary = pd.DataFrame(
    {
        "metric": [
            "R2",
            "RMSE",
            "MAE",
        ],
        "mean": [
            cv_results["test_R2"].mean(),
            (-cv_results["test_RMSE"]).mean(),
            (-cv_results["test_MAE"]).mean(),
        ],
        "std": [
            cv_results["test_R2"].std(ddof=1),
            (-cv_results["test_RMSE"]).std(ddof=1),
            (-cv_results["test_MAE"]).std(ddof=1),
        ],
    }
)
# CV metriklerinin ortalama ve standart sapma tablosu oluşturulur.


display(cv_summary)
# CV özeti gösterilir.


print("✅ CHECKPOINT 03.30: CV özet tablosu hazır.")

## Hücre 31 — Final çıktı dosya adları

In [ ]:
imputer_filename = (
    f"{TARGET_ID}_feature_imputer.joblib"
)
# Eğitim setinde fit edilen median imputer dosya adı oluşturulur.


model_filename = (
    f"{TARGET_ID}_final_tree_model.joblib"
)
# Eğitilmiş final model dosya adı oluşturulur.


bundle_filename = (
    f"{TARGET_ID}_final_model_bundle.json"
)
# Sonraki notebookun kullanacağı model bundle adı oluşturulur.


train_filename = (
    f"{TARGET_ID}_train_predictions.csv"
)
# Eğitim tahmin CSV adı oluşturulur.


test_filename = (
    f"{TARGET_ID}_test_predictions.csv"
)
# Test tahmin CSV adı oluşturulur.


metrics_filename = (
    f"{TARGET_ID}_final_model_metrics.csv"
)
# Holdout metrik CSV adı oluşturulur.


cv_filename = (
    f"{TARGET_ID}_final_model_5CV.csv"
)
# CV özet CSV adı oluşturulur.


prediction_plot_filename = (
    f"{TARGET_ID}_actual_vs_predicted.png"
)
# Gerçek-tahmin grafik dosya adı oluşturulur.


print("✅ CHECKPOINT 03.31: Final çıktı dosya adları hazır.")

## Hücre 32 — Final çıktı yolları

In [ ]:
imputer_path = DRIVE_ROOT / imputer_filename
# Eğitim setinde fit edilen median imputer için Drive yolu oluşturulur.


model_path = DRIVE_ROOT / model_filename
# Final model Drive yolu oluşturulur.


bundle_path = DRIVE_ROOT / bundle_filename
# Model bundle Drive yolu oluşturulur.


train_path = DRIVE_ROOT / train_filename
# Eğitim tahminleri Drive yolu oluşturulur.


test_path = DRIVE_ROOT / test_filename
# Test tahminleri Drive yolu oluşturulur.


metrics_path = DRIVE_ROOT / metrics_filename
# Holdout metrikleri Drive yolu oluşturulur.


cv_path = DRIVE_ROOT / cv_filename
# CV özeti Drive yolu oluşturulur.


prediction_plot_path = (
    DRIVE_ROOT / prediction_plot_filename
)
# Gerçek-tahmin grafiği Drive yolu oluşturulur.


print("✅ CHECKPOINT 03.32: Final çıktı yolları hazır.")

## Hücre 33 — Eğitim tahmin tablosunun hazırlanması

In [ ]:
train_output = (
    df.iloc[
        train_indices
    ][metadata_columns]
    .copy()
)
# Eğitim satırlarının metadata tablosu oluşturulur.


train_output["actual_target"] = (
    y_train.to_numpy()
)
# Eğitim gerçek target değerleri eklenir.


train_output["predicted_target"] = (
    train_predictions
)
# Eğitim tahminleri eklenir.


train_output["residual"] = (
    y_train.to_numpy()
    - train_predictions
)
# Eğitim artıkları hesaplanır.


train_output["source_row_index"] = (
    train_indices
)
# Eğitim satırlarının kaynak indeksleri saklanır.


print("Train çıktı boyutu:", train_output.shape)
print("✅ CHECKPOINT 03.33: Eğitim tahmin tablosu hazır.")

## Hücre 34 — Test tahmin tablosunun hazırlanması

In [ ]:
test_output = (
    df.iloc[
        test_indices
    ][metadata_columns]
    .copy()
)
# Test satırlarının metadata tablosu oluşturulur.


test_output["actual_target"] = (
    y_test.to_numpy()
)
# Test gerçek target değerleri eklenir.


test_output["predicted_target"] = (
    test_predictions
)
# Test tahminleri eklenir.


test_output["residual"] = (
    y_test.to_numpy()
    - test_predictions
)
# Test artıkları hesaplanır.


test_output["source_row_index"] = (
    test_indices
)
# Test satırlarının kaynak indeksleri saklanır.


print("Test çıktı boyutu:", test_output.shape)
print("✅ CHECKPOINT 03.34: Test tahmin tablosu hazır.")

## Hücre 35 — Model bundle içeriğinin hazırlanması

In [ ]:
model_bundle = {
    "target_id": TARGET_ID,
    "github_feature_path": USED_GITHUB_PATH,
    "github_feature_url": USED_GITHUB_URL,
    "pipeline_signature": PIPELINE_SIGNATURE,
    "target_column": target_column,
    "metadata_columns": metadata_columns,
    "feature_names": feature_columns,
    "feature_imputer_filename": imputer_filename,
    "max_feature_missing_ratio": MAX_FEATURE_MISSING_RATIO,
    "dropped_all_missing_features": all_missing_columns,
    "dropped_high_missing_features": high_missing_columns,
    "dropped_constant_features": constant_columns,
    "imputer_statistics": feature_imputer.statistics_.tolist(),
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "cv_folds": CV_FOLDS,
    "selected_model_name": selected_model_name,
    "selected_model_params": selected_model_params,
    "train_indices": train_indices.tolist(),
    "test_indices": test_indices.tolist(),
    "train_metrics": train_metrics,
    "test_metrics": test_metrics,
}
# Sonraki notebookun aynı feature sırası, split ve modeli kullanmasını sağlayan bundle hazırlanır.


print("✅ CHECKPOINT 03.35: Model bundle içeriği hazır.")

## Hücre 36 — Model ve tablo çıktılarının Drive'a kaydedilmesi

In [ ]:
joblib.dump(
    feature_imputer,
    imputer_path,
)
# Eğitim setinde fit edilen median imputer Google Drive'a kaydedilir.


joblib.dump(
    final_model,
    model_path,
)
# Eğitilmiş final model Google Drive'a kaydedilir.


train_output.to_csv(
    train_path,
    index=False,
)
# Eğitim tahmin tablosu Google Drive'a kaydedilir.


test_output.to_csv(
    test_path,
    index=False,
)
# Test tahmin tablosu Google Drive'a kaydedilir.


metrics_table.to_csv(
    metrics_path,
    index=True,
)
# Holdout metrik tablosu Google Drive'a kaydedilir.


cv_summary.to_csv(
    cv_path,
    index=False,
)
# 5-fold CV özeti Google Drive'a kaydedilir.


with open(
    bundle_path,
    "w",
    encoding="utf-8",
) as file:
    # Model bundle dosyası yazma modunda açılır.

    json.dump(
        model_bundle,
        file,
        ensure_ascii=False,
        indent=2,
    )
    # Model bundle okunabilir JSON biçiminde Drive'a kaydedilir.


print("✅ CHECKPOINT 03.36: Model ve tablo çıktıları Drive'a kaydedildi.")

## Hücre 37 — Gerçek ve tahmin edilen değerlerin çizilmesi

In [ ]:
plt.figure(
    figsize=(7, 7)
)
# Gerçek-tahmin grafiği için yeni figür oluşturulur.


plt.scatter(
    y_train,
    train_predictions,
    alpha=0.55,
    label="Train",
)
# Eğitim gerçek ve tahmin değerleri çizilir.


plt.scatter(
    y_test,
    test_predictions,
    alpha=0.75,
    label="Test",
)
# Test gerçek ve tahmin değerleri çizilir.


axis_min = min(
    y.min(),
    train_predictions.min(),
    test_predictions.min(),
)
# Grafik alt sınırı hesaplanır.


axis_max = max(
    y.max(),
    train_predictions.max(),
    test_predictions.max(),
)
# Grafik üst sınırı hesaplanır.


plt.plot(
    [axis_min, axis_max],
    [axis_min, axis_max],
    linestyle="--",
)
# İdeal y=x doğrusu çizilir.


plt.xlabel("Gerçek target")
# X ekseni etiketi eklenir.


plt.ylabel("Tahmin edilen target")
# Y ekseni etiketi eklenir.


plt.title(
    f"{TARGET_ID} — {selected_model_name}"
)
# Grafik başlığı eklenir.


plt.legend()
# Train ve test açıklaması eklenir.


plt.tight_layout()
# Grafik elemanlarının taşması engellenir.


plt.savefig(
    prediction_plot_path,
    dpi=300,
    bbox_inches="tight",
)
# Grafik Google Drive'a kaydedilir.


plt.show()
# Grafik notebook içinde gösterilir.


print("✅ CHECKPOINT 03.37: Gerçek-tahmin grafiği kaydedildi.")

## Hücre 38 — Bütün çıktıların son kontrolü

In [ ]:
OUTPUT_PATHS = [
    imputer_path,
    lazy_results_path,
    lazy_predictions_path,
    lazy_cache_metadata_path,
    model_path,
    bundle_path,
    train_path,
    test_path,
    metrics_path,
    cv_path,
    prediction_plot_path,
]
# Üretilmesi beklenen bütün Drive çıktı yolları listelenir.


for output_path in OUTPUT_PATHS:
    # Bütün çıktılar sırayla kontrol edilir.

    if not output_path.exists():
        # Dosyanın gerçekten oluşup oluşmadığı kontrol edilir.

        raise IOError(
            f"Drive çıktısı oluşturulamadı: {output_path}"
        )
        # Eksik çıktı varsa pipeline durdurulur.

    if output_path.stat().st_size == 0:
        # Dosyanın boş olup olmadığı kontrol edilir.

        raise IOError(
            f"Drive çıktısı boş oluşturuldu: {output_path}"
        )
        # Boş çıktı kabul edilmez.

    print(
        "[Drive kaydı]",
        output_path,
        f"({output_path.stat().st_size:,} byte)",
    )
    # Başarılı çıktı yolu ve dosya boyutu gösterilir.


print()
print(
    "Sonraki notebook için ana bundle:",
    bundle_filename,
)
print(
    "✅ CHECKPOINT 03.38: Notebook tamamlandı; "
    "bütün çıktılar Google Drive'a kaydedildi."
)

## Notebook sonu

Bu sürümde Mordred matrisindeki eksik değerler nedeniyle modelleme durmaz:

- Tamamen boş descriptorlar çıkarılır.
- Eksik oranı `%20` üzerinde olan descriptorlar çıkarılır.
- Sabit descriptorlar çıkarılır.
- Median imputer yalnızca `X_train` üzerinde fit edilir.
- Aynı imputer `X_test` üzerinde yalnızca transform için kullanılır.
- Final model ve LazyPredict aynı temizlenmiş feature setini kullanır.